# ML パイプライン自動化（Snowflake Task + SPCS推論）

## 概要
SPCSにデプロイ済みの `SALES_FORECAST_SERVICE` を **SQL内から直接呼び出し**、
7日間の売上予測を毎朝自動実行する Task を構築します。

### アプローチ
Python ストアドプロシージャを使わず、**純SQL** でバッチ推論を実行します。
`SALES_FORECAST_SERVICE!PREDICT(...)` でSPCSサービスをSQL関数として呼び出せるため、
1つの INSERT 文で特徴量構築 → 推論 → 結果保存を完結できます。

### パイプライン構成
```
BUYER_AGENT (ソース)                  SALES_ML (自動化)
┌─────────────────────┐        ┌──────────────────────────────────┐
│ ID_POS_TRANSACTIONS │──→     │ DAILY_SALES_FORECAST_TASK        │
│ PRODUCT_MASTER      │        │   直近21日のPOSデータから        │
└─────────────────────┘        │   特徴量構築 → SPCS推論          │
                                │   → 7日間予測を結果テーブルに保存│
                                │          ↓                      │
                                │ DAILY_DRIFT_CHECK_TASK           │
                                │   → Model Monitor ドリフト検知  │
                                └──────────────────────────────────┘
```

### 前提条件
- `FOODEX_DEMO.SALES_ML` スキーマ作成済み
- Model `SALES_FORECAST_MODEL` が Registry 登録済み
- SPCS Service `SALES_FORECAST_SERVICE` がデプロイ済み（READY状態）
- Model Monitor `SALES_FORECAST_MONITOR` 設定済み

## Step 1: 環境セットアップ

In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F

session = get_active_session()
session.sql("USE DATABASE FOODEX_DEMO").collect()
session.sql("USE SCHEMA SALES_ML").collect()
session.sql("USE WAREHOUSE COMPUTE_WH").collect()

print("Database:  FOODEX_DEMO")
print("Schema:    SALES_ML")
print("Warehouse: COMPUTE_WH")

## Step 2: 既存オブジェクトの確認

Task が依存するオブジェクトが全て存在するか確認します。

In [ ]:
print("=== Models ===")
session.sql("SHOW MODELS IN SCHEMA FOODEX_DEMO.SALES_ML").show()

print("\n=== Services ===")
session.sql("SHOW SERVICES IN SCHEMA FOODEX_DEMO.SALES_ML").show()

print("\n=== Model Monitors ===")
session.sql("SHOW MODEL MONITORS IN SCHEMA FOODEX_DEMO.SALES_ML").show()

In [ ]:
%%sql -r service_status
DESCRIBE SERVICE FOODEX_DEMO.SALES_ML.SALES_FORECAST_SERVICE;

## Step 3: 予測結果テーブルの作成

Task が INSERT する先のテーブルを作成します。

| カラム | 型 | 説明 |
|--------|------|------|
| FORECAST_DATE | DATE | 予測対象日（今日+1〜+7） |
| CATEGORY_MEDIUM | VARCHAR | 商品カテゴリ |
| PREDICTED_SALES | FLOAT | 予測売上額 |
| CREATED_AT | TIMESTAMP_NTZ | 予測実行日時 |

In [ ]:
%%sql -r create_table
CREATE TABLE IF NOT EXISTS FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS (
    FORECAST_DATE    DATE            NOT NULL,
    CATEGORY_MEDIUM  VARCHAR(100)    NOT NULL,
    PREDICTED_SALES  FLOAT           NOT NULL,
    CREATED_AT       TIMESTAMP_NTZ   NOT NULL DEFAULT CURRENT_TIMESTAMP()
)
COMMENT = '日次7日間売上予測結果（DAILY_SALES_FORECAST_TASK で毎朝自動INSERT）';

## Step 4: 推論SQLの動作確認

Task に設定する前に、INSERT 文のSELECT部分を単体で実行してテストします。

### 処理フロー（純SQL）:
1. **daily_agg**: 直近21日のPOSデータをカテゴリ×日別に集計
2. **base_features**: LAG/移動平均の特徴量を構築
3. **latest_features**: 最新日の特徴量のみ抽出
4. **category_encoding**: DENSE_RANK でカテゴリをエンコーディング
5. **forecast_days**: 7日分のオフセットを生成
6. **PREDICT**: SPCS サービスで推論実行

In [ ]:
%%sql -r test_query
WITH daily_agg AS (
    SELECT 
        t.TRANSACTION_DATE AS SALES_DATE,
        p.CATEGORY_MEDIUM,
        SUM(t.SALES_AMOUNT) AS DAILY_SALES,
        COUNT(DISTINCT t.TRANSACTION_ID) AS TXN_COUNT,
        SUM(t.QUANTITY) AS TOTAL_QTY
    FROM FOODEX_DEMO.BUYER_AGENT.ID_POS_TRANSACTIONS t
    JOIN FOODEX_DEMO.BUYER_AGENT.PRODUCT_MASTER p 
        ON t.PRODUCT_ID = p.PRODUCT_ID
    WHERE p.CATEGORY_MEDIUM IS NOT NULL
      AND t.TRANSACTION_DATE >= DATEADD('day', -21, CURRENT_DATE())
    GROUP BY t.TRANSACTION_DATE, p.CATEGORY_MEDIUM
),
base_features AS (
    SELECT 
        CATEGORY_MEDIUM,
        LAG(DAILY_SALES, 1) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_1,
        LAG(DAILY_SALES, 2) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_2,
        LAG(DAILY_SALES, 3) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_3,
        LAG(DAILY_SALES, 7) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_7,
        LAG(DAILY_SALES, 14) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_14,
        AVG(DAILY_SALES) OVER (
            PARTITION BY CATEGORY_MEDIUM 
            ORDER BY SALES_DATE 
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS MA_7,
        LAG(TXN_COUNT, 1) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS TXN_COUNT,
        LAG(TOTAL_QTY, 1) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS TOTAL_QTY,
        SALES_DATE
    FROM daily_agg
),
latest_features AS (
    SELECT * FROM base_features
    WHERE SALES_DATE = (SELECT MAX(SALES_DATE) FROM daily_agg)
),
category_encoding AS (
    SELECT DISTINCT 
        CATEGORY_MEDIUM,
        DENSE_RANK() OVER (ORDER BY CATEGORY_MEDIUM) - 1 AS CATEGORY_ENCODED
    FROM daily_agg
),
forecast_days AS (
    SELECT SEQ4() + 1 AS DAY_OFFSET
    FROM TABLE(GENERATOR(ROWCOUNT => 7))
)
SELECT 
    DATEADD('day', d.DAY_OFFSET, CURRENT_DATE()) AS FORECAST_DATE,
    f.CATEGORY_MEDIUM,
    FOODEX_DEMO.SALES_ML.SALES_FORECAST_SERVICE!PREDICT(
        f.LAG_1, f.LAG_2, f.LAG_3, f.LAG_7, f.LAG_14, f.MA_7,
        DAYOFWEEK(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())),
        CASE WHEN DAYOFWEEK(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())) IN (0, 6) THEN 1 ELSE 0 END,
        MONTH(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())),
        QUARTER(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())),
        f.TXN_COUNT, f.TOTAL_QTY, c.CATEGORY_ENCODED
    ):PREDICTED_SALES::FLOAT AS PREDICTED_SALES,
    CURRENT_TIMESTAMP() AS CREATED_AT
FROM latest_features f
CROSS JOIN forecast_days d
JOIN category_encoding c ON f.CATEGORY_MEDIUM = c.CATEGORY_MEDIUM
WHERE f.LAG_14 IS NOT NULL
ORDER BY f.CATEGORY_MEDIUM, FORECAST_DATE;

## Step 5: Snowflake Task の作成

### Task チェーン構成
```
DAILY_SALES_FORECAST_TASK (Root, 毎朝AM6時 JST)
  │  直近21日のPOSデータ → 特徴量構築 → SPCS推論
  │  → SALES_FORECAST_RESULTS に7日間予測をINSERT
  ↓
DAILY_DRIFT_CHECK_TASK (Child, 推論完了後)
     Model Monitor の PSI をチェック
```

### ポイント:
- **WAREHOUSE指定**: `COMPUTE_WH` を明示指定（SPCS呼び出しにはWHが必要）
- **スケジュール**: `CRON 0 6 * * * Asia/Tokyo`（毎朝6時JST）
- **純SQL**: Python不要、ストアドプロシージャ不要

In [ ]:
%%sql -r task_forecast
CREATE OR REPLACE TASK FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK
    WAREHOUSE = COMPUTE_WH
    SCHEDULE = 'USING CRON 0 6 * * * Asia/Tokyo'
    COMMENT = '毎朝6時に7日間の売上予測を実行（SPCS経由）'
AS
INSERT INTO FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS
WITH daily_agg AS (
    SELECT 
        t.TRANSACTION_DATE AS SALES_DATE,
        p.CATEGORY_MEDIUM,
        SUM(t.SALES_AMOUNT) AS DAILY_SALES,
        COUNT(DISTINCT t.TRANSACTION_ID) AS TXN_COUNT,
        SUM(t.QUANTITY) AS TOTAL_QTY
    FROM FOODEX_DEMO.BUYER_AGENT.ID_POS_TRANSACTIONS t
    JOIN FOODEX_DEMO.BUYER_AGENT.PRODUCT_MASTER p 
        ON t.PRODUCT_ID = p.PRODUCT_ID
    WHERE p.CATEGORY_MEDIUM IS NOT NULL
      AND t.TRANSACTION_DATE >= DATEADD('day', -21, CURRENT_DATE())
    GROUP BY t.TRANSACTION_DATE, p.CATEGORY_MEDIUM
),
base_features AS (
    SELECT 
        CATEGORY_MEDIUM,
        LAG(DAILY_SALES, 1) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_1,
        LAG(DAILY_SALES, 2) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_2,
        LAG(DAILY_SALES, 3) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_3,
        LAG(DAILY_SALES, 7) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_7,
        LAG(DAILY_SALES, 14) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_14,
        AVG(DAILY_SALES) OVER (
            PARTITION BY CATEGORY_MEDIUM 
            ORDER BY SALES_DATE 
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS MA_7,
        LAG(TXN_COUNT, 1) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS TXN_COUNT,
        LAG(TOTAL_QTY, 1) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS TOTAL_QTY,
        SALES_DATE
    FROM daily_agg
),
latest_features AS (
    SELECT * FROM base_features
    WHERE SALES_DATE = (SELECT MAX(SALES_DATE) FROM daily_agg)
),
category_encoding AS (
    SELECT DISTINCT 
        CATEGORY_MEDIUM,
        DENSE_RANK() OVER (ORDER BY CATEGORY_MEDIUM) - 1 AS CATEGORY_ENCODED
    FROM daily_agg
),
forecast_days AS (
    SELECT SEQ4() + 1 AS DAY_OFFSET
    FROM TABLE(GENERATOR(ROWCOUNT => 7))
)
SELECT 
    DATEADD('day', d.DAY_OFFSET, CURRENT_DATE()) AS FORECAST_DATE,
    f.CATEGORY_MEDIUM,
    FOODEX_DEMO.SALES_ML.SALES_FORECAST_SERVICE!PREDICT(
        f.LAG_1, f.LAG_2, f.LAG_3, f.LAG_7, f.LAG_14, f.MA_7,
        DAYOFWEEK(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())),
        CASE WHEN DAYOFWEEK(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())) IN (0, 6) THEN 1 ELSE 0 END,
        MONTH(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())),
        QUARTER(DATEADD('day', d.DAY_OFFSET, CURRENT_DATE())),
        f.TXN_COUNT, f.TOTAL_QTY, c.CATEGORY_ENCODED
    ):PREDICTED_SALES::FLOAT AS PREDICTED_SALES,
    CURRENT_TIMESTAMP() AS CREATED_AT
FROM latest_features f
CROSS JOIN forecast_days d
JOIN category_encoding c ON f.CATEGORY_MEDIUM = c.CATEGORY_MEDIUM
WHERE f.LAG_14 IS NOT NULL;

### ドリフト検知 Child Task

推論完了後に Model Monitor の PSI を確認し、ドリフトを検知します。

In [ ]:
%%sql -r sp_drift
CREATE OR REPLACE PROCEDURE FOODEX_DEMO.SALES_ML.SP_CHECK_MODEL_DRIFT()
    RETURNS VARCHAR
    LANGUAGE SQL
AS
$$
DECLARE
    drift_count INTEGER;
    max_psi FLOAT;
    result_msg VARCHAR;
BEGIN
    SELECT COUNT(*), COALESCE(MAX(METRIC_VALUE), 0)
    INTO :drift_count, :max_psi
    FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
        'SALES_FORECAST_MONITOR',
        'PSI',
        'LAG_1',
        'DAY',
        DATEADD('day', -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
        CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
    ))
    WHERE METRIC_VALUE >= 0.25;

    IF (:drift_count > 0) THEN
        result_msg := 'ALERT: Critical drift detected! PSI=' || :max_psi::VARCHAR || ' (' || :drift_count::VARCHAR || ' days). Consider model retraining.';
    ELSE
        SELECT COALESCE(MAX(METRIC_VALUE), 0) INTO :max_psi
        FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
            'SALES_FORECAST_MONITOR',
            'PSI',
            'LAG_1',
            'DAY',
            DATEADD('day', -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
            CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
        ));
        result_msg := 'OK: No critical drift. Max PSI=' || :max_psi::VARCHAR;
    END IF;

    RETURN result_msg;
END;
$$;

In [ ]:
%%sql -r task_drift
CREATE OR REPLACE TASK FOODEX_DEMO.SALES_ML.DAILY_DRIFT_CHECK_TASK
    USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE = 'XSMALL'
    COMMENT = '推論完了後にModel Monitorのドリフト検知を実行'
    AFTER FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK
AS
    CALL FOODEX_DEMO.SALES_ML.SP_CHECK_MODEL_DRIFT();

## Step 6: Task の有効化

Task チェーンでは **Child → Root** の順に RESUME する必要があります。

**注意**: 有効化するとスケジュールに従って自動実行が開始されます。

In [ ]:
%%sql -r resume_child
ALTER TASK FOODEX_DEMO.SALES_ML.DAILY_DRIFT_CHECK_TASK RESUME;

In [ ]:
%%sql -r resume_root
ALTER TASK FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK RESUME;

In [ ]:
print("=== Task 一覧 ===")
session.sql("SHOW TASKS IN SCHEMA FOODEX_DEMO.SALES_ML").show()

## Step 7: 手動実行テスト

`EXECUTE TASK` でスケジュールを待たずに即時実行できます。
Root Task を実行すると、完了後に Child Task (ドリフト検知) も自動実行されます。

In [ ]:
%%sql -r exec_result
EXECUTE TASK FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK;

In [ ]:
import time

print("Task 実行状況を確認中...")
for i in range(12):
    time.sleep(10)
    result = session.sql("""
        SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME, ERROR_MESSAGE
        FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
            TASK_NAME => 'DAILY_SALES_FORECAST_TASK',
            SCHEDULED_TIME_RANGE_START => DATEADD('hour', -1, CURRENT_TIMESTAMP())
        ))
        ORDER BY SCHEDULED_TIME DESC
        LIMIT 1
    """).to_pandas()

    if len(result) > 0:
        state = result.iloc[0]['STATE']
        print(f"  [{i+1}/12] State: {state}")
        if state == 'SUCCEEDED':
            print(f"\nTask completed successfully!")
            break
        elif state == 'FAILED':
            print(f"\nTask failed!")
            print(f"  Error: {result.iloc[0]['ERROR_MESSAGE']}")
            break
    else:
        print(f"  [{i+1}/12] Waiting...")
else:
    print("\nTask still running. Check history later.")

## Step 8: 予測結果の確認

In [ ]:
%%sql -r latest_results
SELECT *
FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS
WHERE CREATED_AT = (SELECT MAX(CREATED_AT) FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS)
ORDER BY CATEGORY_MEDIUM, FORECAST_DATE;

In [ ]:
print("=== 予測結果サマリー ===")
session.sql("""
    SELECT 
        CREATED_AT::DATE AS BATCH_DATE,
        COUNT(*) AS TOTAL_RECORDS,
        COUNT(DISTINCT CATEGORY_MEDIUM) AS CATEGORIES,
        MIN(FORECAST_DATE) AS FORECAST_FROM,
        MAX(FORECAST_DATE) AS FORECAST_TO,
        ROUND(SUM(PREDICTED_SALES), 0) AS TOTAL_PREDICTED_SALES
    FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS
    GROUP BY BATCH_DATE
    ORDER BY BATCH_DATE DESC
    LIMIT 7
""").show()

In [ ]:
import matplotlib.pyplot as plt

latest_forecast = session.sql("""
    SELECT FORECAST_DATE, CATEGORY_MEDIUM, PREDICTED_SALES
    FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS
    WHERE CREATED_AT = (SELECT MAX(CREATED_AT) FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS)
    ORDER BY CATEGORY_MEDIUM, FORECAST_DATE
""").to_pandas()

if len(latest_forecast) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    categories = latest_forecast['CATEGORY_MEDIUM'].unique()
    for cat in categories:
        cat_data = latest_forecast[latest_forecast['CATEGORY_MEDIUM'] == cat]
        axes[0].plot(cat_data['FORECAST_DATE'], cat_data['PREDICTED_SALES'],
                     marker='o', label=cat, linewidth=2)
    axes[0].set_xlabel('Forecast Date')
    axes[0].set_ylabel('Predicted Sales (円)')
    axes[0].set_title('7日間売上予測（カテゴリ別）')
    axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
    axes[0].grid(alpha=0.3)
    axes[0].tick_params(axis='x', rotation=45)

    cat_totals = latest_forecast.groupby('CATEGORY_MEDIUM')['PREDICTED_SALES'].sum().sort_values(ascending=True)
    axes[1].barh(cat_totals.index, cat_totals.values, color='steelblue')
    axes[1].set_xlabel('7日間合計予測売上 (円)')
    axes[1].set_title('カテゴリ別 7日間合計予測')
    axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()
else:
    print("予測結果がINSERTされるとグラフが表示されます。")

## Step 9: Task 実行履歴の確認

In [ ]:
%%sql -r history_forecast
SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME,
       DATEDIFF('second', QUERY_START_TIME, COMPLETED_TIME) AS DURATION_SEC,
       ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    TASK_NAME => 'DAILY_SALES_FORECAST_TASK',
    SCHEDULED_TIME_RANGE_START => DATEADD('day', -7, CURRENT_TIMESTAMP())
))
ORDER BY SCHEDULED_TIME DESC
LIMIT 10;

In [ ]:
%%sql -r history_drift
SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME,
       RETURN_VALUE, ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    TASK_NAME => 'DAILY_DRIFT_CHECK_TASK',
    SCHEDULED_TIME_RANGE_START => DATEADD('day', -7, CURRENT_TIMESTAMP())
))
ORDER BY SCHEDULED_TIME DESC
LIMIT 10;

## Step 10: 古い予測結果のクリーンアップ（オプション）

過去の予測結果が蓄積されるため、定期的にクリーンアップするTaskも設定できます。

In [ ]:
%%sql -r cleanup_task
CREATE OR REPLACE TASK FOODEX_DEMO.SALES_ML.WEEKLY_FORECAST_CLEANUP_TASK
    USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE = 'XSMALL'
    SCHEDULE = 'USING CRON 0 0 * * 0 Asia/Tokyo'
    COMMENT = '毎週日曜深夜に30日以上前の予測結果を削除'
AS
    DELETE FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS
    WHERE CREATED_AT < DATEADD('day', -30, CURRENT_TIMESTAMP());

In [ ]:
-- 必要に応じて有効化
-- ALTER TASK FOODEX_DEMO.SALES_ML.WEEKLY_FORECAST_CLEANUP_TASK RESUME;

## 運用コマンド集

### Task の一時停止（Root → Child の順）
```sql
ALTER TASK FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK SUSPEND;
ALTER TASK FOODEX_DEMO.SALES_ML.DAILY_DRIFT_CHECK_TASK SUSPEND;
```

### Task の再開（Child → Root の順）
```sql
ALTER TASK FOODEX_DEMO.SALES_ML.DAILY_DRIFT_CHECK_TASK RESUME;
ALTER TASK FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK RESUME;
```

### Task の即時実行
```sql
EXECUTE TASK FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK;
```

### 最新の予測結果を確認
```sql
SELECT * FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS
WHERE CREATED_AT = (SELECT MAX(CREATED_AT) FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS)
ORDER BY CATEGORY_MEDIUM, FORECAST_DATE;
```

### Task 実行履歴
```sql
SELECT * FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    TASK_NAME => 'DAILY_SALES_FORECAST_TASK',
    SCHEDULED_TIME_RANGE_START => DATEADD('day', -7, CURRENT_TIMESTAMP())
)) ORDER BY SCHEDULED_TIME DESC;
```

### Task の削除
```sql
ALTER TASK FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK SUSPEND;
DROP TASK IF EXISTS FOODEX_DEMO.SALES_ML.DAILY_DRIFT_CHECK_TASK;
DROP TASK IF EXISTS FOODEX_DEMO.SALES_ML.DAILY_SALES_FORECAST_TASK;
DROP TASK IF EXISTS FOODEX_DEMO.SALES_ML.WEEKLY_FORECAST_CLEANUP_TASK;
```

## まとめ

### 作成したオブジェクト

| オブジェクト | 名前 | 説明 |
|------------|------|------|
| Table | SALES_FORECAST_RESULTS | 7日間売上予測結果テーブル |
| Stored Procedure | SP_CHECK_MODEL_DRIFT | ドリフト検知（PSI閾値チェック） |
| Task (Root) | DAILY_SALES_FORECAST_TASK | 毎朝6時(JST) SPCS推論 → INSERT |
| Task (Child) | DAILY_DRIFT_CHECK_TASK | 推論完了後にドリフト検知 |
| Task (独立) | WEEKLY_FORECAST_CLEANUP_TASK | 毎週日曜 古い予測結果を削除 |

### 自動化パイプライン全体像

```
毎時         BUYER_AGENT.TASK_INSERT_SALES_DATA
               POSデータ投入
  ↓
毎朝6時     SALES_ML.DAILY_SALES_FORECAST_TASK
               直近21日POS → 特徴量構築(SQL)
               → SALES_FORECAST_SERVICE!PREDICT (SPCS推論)
               → SALES_FORECAST_RESULTS に7日間予測INSERT
  ↓
完了後       SALES_ML.DAILY_DRIFT_CHECK_TASK
               → SP_CHECK_MODEL_DRIFT (PSIチェック)
  ↓
毎日         SALES_ML.SALES_FORECAST_MONITOR (自動パフォーマンス監視)
```

### Snowsight での確認
- **予測結果**: `SELECT * FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS`
- **Task 履歴**: `Monitoring → Task History`
- **Model Monitor**: `AI&ML → Models → SALES_FORECAST_MODEL → Monitors`

In [ ]:
print("=" * 60)
print("  ML Pipeline Automation Setup Complete")
print("=" * 60)
print(f"\n[推論方式]")
print(f"  純SQL + SPCS Service (SALES_FORECAST_SERVICE!PREDICT)")
print(f"  Python ストアドプロシージャ不要")
print(f"\n[Task チェーン]")
print(f"  DAILY_SALES_FORECAST_TASK : 毎朝6時(JST) [Root]")
print(f"    → 直近21日POS → 特徴量構築 → SPCS推論 → 7日間予測INSERT")
print(f"  DAILY_DRIFT_CHECK_TASK    : 推論完了後    [Child]")
print(f"    → Model Monitor PSI チェック")
print(f"\n[結果テーブル]")
print(f"  FOODEX_DEMO.SALES_ML.SALES_FORECAST_RESULTS")
print(f"\n[全オブジェクト所在]")
print(f"  スキーマ: FOODEX_DEMO.SALES_ML")